In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# 3.1 EXTRAÇÃO
# ==============================================================================
print("--> Carregando dados brutos...")
df_censo = pd.read_csv("censo_escolar_2023.csv", sep=";", encoding="utf-8", low_memory=False)
df_ideb = pd.read_csv("ideb_ensino_medio_2023.csv", sep=";", encoding="utf-8", low_memory=False)

try:
    df_inse = pd.read_excel("INSE_2023_escolas.xlsx", sheet_name="INSE_ESC_2023")
except Exception:
    df_inse = pd.read_csv("INSE_2023_escolas.csv", sep=";", encoding="utf-8", low_memory=False)

# ==============================================================================
# 3.2 TRANSFORMAÇÃO
# ==============================================================================
print("--> Executando transformações...")

# --- A. Censo Escolar (Define o Universo de Ensino Médio) ---
df_censo_tratado = df_censo[df_censo['IN_MED'] == 1].copy()
df_censo_tratado = df_censo_tratado[df_censo_tratado['TP_SITUACAO_FUNCIONAMENTO'] == 1]
cols_quant = [c for c in df_censo_tratado.columns if c.startswith('QT_')]
df_censo_tratado[cols_quant] = df_censo_tratado[cols_quant].replace(88888, np.nan)

# Conjunto de IDs válidos do Ensino Médio
ids_ensino_medio = set(df_censo_tratado['CO_ENTIDADE'])

# --- B. IDEB ---
df_ideb_tratado = df_ideb.copy()
df_ideb_tratado.rename(columns={'ID_ESCOLA': 'CO_ENTIDADE'}, inplace=True)

# Filtro de escopo: reter apenas escolas de Ensino Médio ativas no Censo
df_ideb_tratado = df_ideb_tratado[df_ideb_tratado['CO_ENTIDADE'].isin(ids_ensino_medio)].copy()

cols_notas = ['VL_NOTA_MATEMATICA_2023', 'VL_NOTA_PORTUGUES_2023', 'VL_OBSERVADO_2023']
for col in cols_notas:
    df_ideb_tratado[col] = df_ideb_tratado[col].astype(str).str.replace(',', '.')
    df_ideb_tratado[col] = pd.to_numeric(df_ideb_tratado[col], errors='coerce')

df_ideb_tratado.drop(columns=['VL_APROVACAO_2023_4'], inplace=True, errors='ignore')

# --- C. INSE ---
df_inse_tratado = df_inse.copy()
df_inse_tratado.rename(columns={'ID_ESCOLA': 'CO_ENTIDADE'}, inplace=True)

# Filtro de escopo: remover escolas que não ofertam Ensino Médio
df_inse_tratado = df_inse_tratado[df_inse_tratado['CO_ENTIDADE'].isin(ids_ensino_medio)].copy()

cols_pc = [c for c in df_inse_tratado.columns if c.startswith('PC_NIVEL_')]
df_inse_tratado[cols_pc] = df_inse_tratado[cols_pc].fillna(0)

# ==============================================================================
# 3.3 CARGA
# ==============================================================================
print("--> Salvando datasets tratados...")
df_censo_tratado.to_csv("censo_escolar.csv", index=False, sep=";", encoding="utf-8-sig")
df_ideb_tratado.to_csv("ideb_ensino_medio.csv", index=False, sep=";", encoding="utf-8-sig")
df_inse_tratado.to_csv("inse_escolas.csv", index=False, sep=";", encoding="utf-8-sig")

print(f"censo_escolar:   {df_censo_tratado.shape}")
print(f"ideb_ensino_medio:{df_ideb_tratado.shape}")
print(f"inse_escolas:    {df_inse_tratado.shape}")

--> Carregando dados brutos...
--> Executando transformações...
--> Salvando datasets tratados...
censo_escolar:   (29754, 36)
ideb_ensino_medio:(21060, 15)
inse_escolas:    (19782, 22)
